## K-means

In [ ]:
import numpy as np
class KMeans:
    def __init__(self, k, method='random', max_iter=300):
        self.k = k
        self.method = method
        self.max_iter = max_iter
        self.centroids = None

    def init_centers(self, X):
        if self.method == 'random':
            return X[np.random.choice(X, self.k)] # fix me
        if self.method == 'k-means++':
            centers = [X[np.random.choice(X.shape[0])]]
            for _ in range(1, self.k):
                distances = np.min([np.sum((X - c)**2, axis=1) for c in centers], axis=0)
                probs = distances / distances.sum()
                next_center = X[np.random.choice(X.shape[0], p=probs)]
                centers.append(next_center)
            return np.array(centers)

    def fit(self, X):
        self.centroids = self.init_centers(X)
        for _ in range(self.max_iter):
            old_centroids = self.centroids.copy()
            clusters = self.assign(X, self.centroids)
            self.centroids = self.calculate_centroids(X, clusters)
            if np.all(old_centroids == self.centroids):
                break

    def assign(self, X, centroids):
        distances = np.linalg.norm(X[:, np.newaxis] - centroids, axis=2)
        return np.argmin(distances, axis=1)

    def calculate_centroids(self, X, clusters):
        new_centroids = np.array([X[clusters == i].mean(axis=0) for i in range(self.k)])
        return new_centroids

    def predict(self, X):
        return self.assign(X, self.centroids)



## GMMs

In [ ]:
import pandas as pd
from scipy.stats import multivariate_normal
from typing import Tuple


class GMM:
    def __init__(self, k=4, init='random', max_iter=300):
        self.k = k
        self.max_iter = max_iter
        self.init = init
        self.means = None
        self.covs = None
        self.proportions = None

    def init_params(self, X:  pd.DataFrame) -> Tuple:
        if self.init == 'random':
            indices = np.random.permutation(n_samples)
            X_shuffled = X_arr[indices]
            split_indices = np.array_split(X_shuffled, self.k)

            means = np.array([chunk.mean(axis=0) for chunk in split_indices])
            covs = np.array([np.cov(chunk, rowvar=False) for chunk in split_indices])
            proportions = np.ones(self.k) / self.k
            return means, covs, proportions

        if self.init == 'kmeans':
            kmeans = KMeans(k=self.k)
            kmeans.fit(X_arr)
            labels = kmeans.predict(X_arr)

            means = []
            covs = []
            proportions = []

            for i in range(self.k):
                cluster_data = X_arr[labels == i]
                means.append(cluster_data.mean(axis=0))
                covs.append(np.cov(cluster_data, rowvar=False))
                proportions.append(len(cluster_data) / n_samples)

            return np.array(means), np.array(covs), np.array(proportions)

    def fit(self, X):
        self.means, self.covs, self.proportions = self.init_params(X)
        for _ in range(self.max_iter):
            gamma_mtrx = self.expectation(X, self.means, self.covs, self.proportions)
            means, covs, proportions = self.maximization(X, gamma_mtrx)

            if np.allclose(means, self.means) and np.allclose(covs, self.covs):
                self.means = means
                self.covs = covs
                self.proportions = proportions
                break

            self.means = means
            self.covs = covs
            self.proportions = proportions

    def expectation(self, X, means, covs, proportions):
        X_arr = X.values if isinstance(X, pd.DataFrame) else X
        n_samples = X_arr.shape[0]
        gamma_mtrx = np.zeros((n_samples, self.k))

        for i in range(self.k):
            likelihood = multivariate_normal.pdf(X_arr, mean=means[i], cov=covs[i], allow_singular=True)
            gamma_mtrx[:, i] = proportions[i] * likelihood

        sum_gamma = gamma_mtrx.sum(axis=1, keepdims=True)
        return gamma_mtrx / sum_gamma


    def maximization(self, X, gamma_mtrx):
        X_arr = X.values if isinstance(X, pd.DataFrame) else X
        n_samples, n_features = X_arr.shape

        Nk = gamma_mtrx.sum(axis=0)
        proportions = Nk / n_samples
        means = (gamma_mtrx.T @ X_arr) / Nk[:, np.newaxis]

        covs = np.empty((self.k, n_features, n_features))
        for i in range(self.k):
            diff = X_arr - means[i]
            covs[i] = (gamma_mtrx[:, i] * diff.T @ diff) / Nk[i]

        return means, covs, proportions

    def predict_proba(self, X):
        return self.expectation(X, self.means, self.covs, self.proportions)

    def predict(self, X):
        probabilities = self.predict_proba(X)
        return np.argmax(probabilities, axis=1)
